# V7 Second-Backbone Experiment: CosPlace ResNet18/512

This notebook performs the reviewer-requested **second visual backbone** experiment without changing the language branch or ground truth.

Design:
- Second backbone: **CosPlace ResNet18, 512-D**
- Image size: **320×320**
- Language matrices and positive matrices: reuse the locked files in `vpr_research/embeddings`
- Source calibration: **MSLS 222-query calibration split only, seed 42**
- Target labels from AmsterTime/Nordland are never used for parameter selection
- PWF, tuned constant-α Top-10, and tuned SG are calibrated independently for the CosPlace backbone
- Final evaluation: MSLS held-out, AmsterTime, Nordland-aligned


In [ ]:
# 1) Mount Drive and check GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, glob, json, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

print("Python:", sys.version)
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch check failed:", e)

ROOT = Path('/content/drive/MyDrive/vpr_research')
EMBED = ROOT / 'embeddings'
OUT = ROOT / 'acra_final_results' / 'v7_cosplace_second_backbone'
OUT.mkdir(parents=True, exist_ok=True)
print("Output:", OUT)


In [ ]:
# 2) Setup auto_VPR and official AmsterTime downloader
%cd /content

if not Path('/content/auto_VPR').exists():
    !git clone --recursive https://github.com/gmberton/auto_VPR
%cd /content/auto_VPR
!git submodule update --init --recursive
!pip install -q -r requirements.txt
!pip install -q faiss-cpu

%cd /content
if not Path('/content/VPR-datasets-downloader').exists():
    !git clone https://github.com/gmberton/VPR-datasets-downloader
%cd /content/VPR-datasets-downloader
!pip install -q -r requirements.txt

AMSTER_DB = Path('/content/VPR-datasets-downloader/datasets/amstertime/images/test/database')
AMSTER_Q  = Path('/content/VPR-datasets-downloader/datasets/amstertime/images/test/queries')

if not AMSTER_DB.exists() or not AMSTER_Q.exists():
    !python download_amstertime.py

print("AmsterTime DB:", len(list(AMSTER_DB.glob('*'))))
print("AmsterTime Q :", len(list(AMSTER_Q.glob('*'))))


In [ ]:
# 3) Prepare MSLS and Nordland image folders

MSLS_ZIP = ROOT / 'msls_raw' / 'msls_val_subset.zip'
MSLS_EXTRACT = Path('/content/msls_val_subset')

if not MSLS_EXTRACT.exists():
    print("Extracting", MSLS_ZIP)
    with zipfile.ZipFile(MSLS_ZIP, 'r') as z:
        z.extractall(MSLS_EXTRACT)

def find_named_dir(root, name):
    hits = [p for p in Path(root).rglob(name) if p.is_dir()]
    if not hits:
        raise FileNotFoundError(f"Could not find a directory named {name!r} under {root}")
    # Prefer paths containing 'test'
    hits = sorted(hits, key=lambda p: (0 if 'test' in str(p).lower() else 1, len(str(p))))
    return hits[0]

MSLS_DB = find_named_dir(MSLS_EXTRACT, 'database')
MSLS_Q  = find_named_dir(MSLS_EXTRACT, 'queries')

NORD_DB = ROOT / 'nordland_clean_subset' / 'images' / 'test' / 'database'
NORD_Q  = ROOT / 'nordland_clean_subset' / 'images' / 'test' / 'queries'

for name, db, q in [
    ('AmsterTime', AMSTER_DB, AMSTER_Q),
    ('MSLS', MSLS_DB, MSLS_Q),
    ('Nordland', NORD_DB, NORD_Q),
]:
    print(name)
    print("  DB:", db, len(list(Path(db).glob('*'))))
    print("  Q :", q, len(list(Path(q).glob('*'))))


In [ ]:
# 4) Run CosPlace ResNet18/512 and save descriptors

AUTO = Path('/content/auto_VPR')
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
BATCH = 64 if DEVICE == 'cuda' else 8

def run_cosplace(name, db_dir, q_dir):
    log_name = f'v7_cosplace_{name}'
    base = AUTO / 'logs' / log_name

    # Reuse completed run if descriptors already exist.
    existing = sorted(base.glob('*/database_descriptors.npy'))
    existing_q = sorted(base.glob('*/queries_descriptors.npy'))
    if existing and existing_q:
        print(f"{name}: reusing existing descriptors")
        return existing[-1].parent

    cmd = (
        f"cd {AUTO} && "
        f"printf 'y\\n' | python main.py "
        f"--method=cosplace "
        f"--backbone=ResNet18 "
        f"--descriptors_dimension=512 "
        f"--database_folder='{db_dir}' "
        f"--queries_folder='{q_dir}' "
        f"--device={DEVICE} "
        f"--image_size 320 320 "
        f"--batch_size {BATCH} "
        f"--num_workers 2 "
        f"--recall_values 1 5 10 "
        f"--save_descriptors "
        f"--no_labels "
        f"--log_dir {log_name}"
    )
    print(cmd)
    subprocess.run(cmd, shell=True, check=True)

    runs = sorted(base.glob('*'))
    if not runs:
        raise RuntimeError(f"No auto_VPR output found for {name}")
    return runs[-1]

runs = {}
runs['amstertime'] = run_cosplace('amstertime', AMSTER_DB, AMSTER_Q)
runs['msls']       = run_cosplace('msls', MSLS_DB, MSLS_Q)
runs['nordland']   = run_cosplace('nordland', NORD_DB, NORD_Q)

print(runs)


In [ ]:
# 5) Convert descriptors to CosPlace visual similarity matrices and verify ordering

def descriptor_files(run_dir):
    db = list(Path(run_dir).rglob('database_descriptors.npy'))
    q  = list(Path(run_dir).rglob('queries_descriptors.npy'))
    if not db or not q:
        raise FileNotFoundError(f"Descriptor files not found in {run_dir}")
    return db[0], q[0]

def make_sim(run_dir):
    dbf, qf = descriptor_files(run_dir)
    db = np.load(dbf).astype(np.float32)
    q = np.load(qf).astype(np.float32)

    # Explicit L2 normalization so dot product is cosine similarity.
    db /= np.linalg.norm(db, axis=1, keepdims=True) + 1e-12
    q  /= np.linalg.norm(q,  axis=1, keepdims=True) + 1e-12
    sim = q @ db.T
    return sim.astype(np.float32)

cos_visual = {d: make_sim(run) for d, run in runs.items()}

for d, sim in cos_visual.items():
    out = OUT / f'{d}_cosplace_visual_sim_matrix.npy'
    np.save(out, sim)
    print(d, sim.shape, 'saved ->', out)


In [ ]:
# 6) Load locked language / positive matrices and the locked MSLS split

FILES = {
    'amstertime': (
        EMBED / 'amstertime_lang_sim_matrix.npy',
        EMBED / 'amstertime_positive_matrix.npy'
    ),
    'msls': (
        EMBED / 'msls_val_lang_sim_matrix.npy',
        EMBED / 'msls_val_positive_matrix.npy'
    ),
    'nordland': (
        EMBED / 'nordland_clean_lang_sim_matrix.npy',
        EMBED / 'nordland_clean_positive_matrix.npy'
    ),
}

language = {}
positive = {}
for d,(lf,pf) in FILES.items():
    language[d] = np.load(lf).astype(np.float32)
    positive[d] = np.load(pf).astype(bool)
    assert cos_visual[d].shape == language[d].shape == positive[d].shape, (
        d, cos_visual[d].shape, language[d].shape, positive[d].shape
    )
    assert positive[d].any(axis=1).all(), f"{d}: query with no positive"
    print(d, cos_visual[d].shape)

split = np.load(ROOT / 'acra_final_results' / 'msls_calibration_split_indices.npz')
CAL = split['calibration_indices']
EVAL = split['evaluation_indices']
print("MSLS calibration:", len(CAL), "held-out:", len(EVAL), "seed:", split['seed'])

# Tie-safe top-k: stable descending sort.
def topk_idx(scores, k):
    return np.argsort(-scores, axis=1, kind='stable')[:, :k]

def correct_at_k(retrieved, pos, k):
    return np.any(np.take_along_axis(pos, retrieved[:, :k], axis=1), axis=1)

def recalls(retrieved, pos):
    return tuple(100.0 * correct_at_k(retrieved, pos, k).mean() for k in (1,5,10))

# CosPlace baseline sanity check.
for d in ['amstertime','msls','nordland']:
    r = recalls(topk_idx(cos_visual[d], 10), positive[d])
    print(d, "CosPlace visual:", r)

# Known AmsterTime CosPlace RN18/512 reference from the earlier run is ~33.6 R@1.
am_r1 = recalls(topk_idx(cos_visual['amstertime'],10), positive['amstertime'])[0]
if abs(am_r1 - 33.6) > 0.75:
    raise RuntimeError(
        f"AmsterTime CosPlace R@1={am_r1:.2f}, far from the earlier ~33.6 reference. "
        "Stop and check image ordering / preprocessing before using the second-backbone results."
    )


In [ ]:
# 7) Exact PWF + fair tuned controls for the CosPlace backbone

EPS = 1e-8
RHO_GRID = [1.0,1.5,2.0,3.0,4.0,6.0,8.0,10.0,12.0,16.0,20.0,30.0]
MC_GRID = [10,20,50,100]
KU = 10
LAMBDA = 0.5

def uncertainty(sorted_scores):
    best = sorted_scores[:, [0]]
    rs = np.mean(sorted_scores[:,1:] / (best + EPS), axis=1)
    sd = np.median(sorted_scores, axis=1) / (sorted_scores[:,0] + EPS)
    return LAMBDA*rs + (1-LAMBDA)*sd

def pwf_retrieve(V, L, rho=3.0, mc=10, norm_stats=None):
    idx = topk_idx(V, mc)
    v = np.take_along_axis(V, idx, axis=1)
    l = np.take_along_axis(L, idx, axis=1)

    # Uncertainty is always K_u=10, independent of M_c.
    vu = v[:, :KU]
    lu_scores = np.sort(l[:, :KU], axis=1)[:, ::-1]
    su = uncertainty(vu)
    lu = uncertainty(lu_scores)

    if norm_stats is None:
        stats = (su.mean(), su.std(), lu.mean(), lu.std())
    else:
        stats = norm_stats

    ms, ss, ml, sl = stats
    suz = (su-ms)/(ss+EPS)
    luz = (lu-ml)/(sl+EPS)
    sv = np.logaddexp(0, suz) + EPS
    lv = np.logaddexp(0, luz) + EPS
    alpha = rho*lv/(sv + rho*lv + EPS)

    fused = alpha[:,None]*v + (1-alpha[:,None])*l
    order = np.argsort(-fused, axis=1, kind='stable')
    return np.take_along_axis(idx, order, axis=1), alpha

def const_top10(V, L, alpha):
    idx = topk_idx(V, 10)
    v = np.take_along_axis(V, idx, axis=1)
    l = np.take_along_axis(L, idx, axis=1)
    fused = alpha*v + (1-alpha)*l
    order = np.argsort(-fused, axis=1, kind='stable')
    return np.take_along_axis(idx, order, axis=1)

def sg_retrieve(V, L, tau, theta):
    idx = topk_idx(V, 10)
    v = np.take_along_axis(V, idx, axis=1)
    l = np.take_along_axis(L, idx, axis=1)
    su = uncertainty(v)
    z = (su-su.mean())/(su.std()+EPS)
    alpha = 1.0/(1.0 + np.exp(z/tau))
    alpha = np.where(l.std(axis=1) >= theta, alpha, 1.0)
    fused = alpha[:,None]*v + (1-alpha[:,None])*l
    order = np.argsort(-fused, axis=1, kind='stable')
    return np.take_along_axis(idx, order, axis=1)

# ---- PWF calibration on 222 MSLS calibration queries only ----
Vcal = cos_visual['msls'][CAL]
Lcal = language['msls'][CAL]
Pcal = positive['msls'][CAL]

pgrid=[]
for rho in RHO_GRID:
    for mc in MC_GRID:
        ret,_ = pwf_retrieve(Vcal,Lcal,rho=rho,mc=mc)
        r1,r5,r10 = recalls(ret,Pcal)
        pgrid.append([rho,mc,r1,r5,r10])
pgrid = pd.DataFrame(pgrid, columns=['rho','Mc','R1','R5','R10'])

# Locked rule: maximise R@1; tie -> smaller candidate pool; tie -> smaller rho.
best_r1 = pgrid.R1.max()
cand = pgrid[pgrid.R1 == best_r1].sort_values(['Mc','rho'])
rho_star = float(cand.iloc[0].rho)
mc_star = int(cand.iloc[0].Mc)
print("CosPlace PWF selected:", rho_star, mc_star)

# ---- Fine tuned constant-alpha Top10 ----
arows=[]
for a in np.round(np.arange(0,1.0001,0.01),2):
    ret=const_top10(Vcal,Lcal,float(a))
    r1,r5,r10=recalls(ret,Pcal)
    arows.append([a,r1,r5,r10])
agrid=pd.DataFrame(arows,columns=['alpha','R1','R5','R10'])
best=agrid.R1.max()
# Conservative tie-break: larger visual weight.
alpha_star=float(agrid[agrid.R1==best].sort_values('alpha',ascending=False).iloc[0].alpha)
print("CosPlace constant alpha selected:", alpha_star)

# ---- Fine tuned SG ----
tau_grid=[0.25,0.5,0.75,1.0,1.5,2.0,3.0,4.0]
theta_grid=np.round(np.arange(0,0.1201,0.005),3)
srows=[]
for tau in tau_grid:
    for theta in theta_grid:
        ret=sg_retrieve(Vcal,Lcal,tau,theta)
        r1,r5,r10=recalls(ret,Pcal)
        srows.append([tau,theta,r1,r5,r10])
sgrid=pd.DataFrame(srows,columns=['tau','theta','R1','R5','R10'])
best=sgrid.R1.max()
cand=sgrid[sgrid.R1==best].copy()
cand['tau_dist']=(cand.tau-1.0).abs()
# Conservative tie-break: larger language-activation threshold, then tau closest to original 1.
sel=cand.sort_values(['theta','tau_dist','tau'],ascending=[False,True,True]).iloc[0]
tau_star=float(sel.tau)
theta_star=float(sel.theta)
print("CosPlace SG selected:", tau_star, theta_star)

pgrid.to_csv(OUT/'cosplace_pwf_calibration_sweep.csv',index=False)
agrid.to_csv(OUT/'cosplace_constant_alpha_sweep.csv',index=False)
sgrid.to_csv(OUT/'cosplace_sg_sweep.csv',index=False)


In [ ]:
# 8) Final second-backbone evaluation and paired bootstrap/BH-FDR

def bh(p):
    p=np.asarray(p,float)
    order=np.argsort(p)
    q=np.empty_like(p)
    prev=1.0
    m=len(p)
    for rank_i in range(m-1,-1,-1):
        idx=order[rank_i]
        val=p[idx]*m/(rank_i+1)
        prev=min(prev,val)
        q[idx]=min(prev,1.0)
    return q

def paired_boot(a,b,B=5000,seed=42):
    a=np.asarray(a); b=np.asarray(b); n=len(a)
    point=100*(a.mean()-b.mean())
    rng=np.random.default_rng(seed)
    diffs=np.empty(B)
    for i in range(B):
        ix=rng.integers(0,n,n)
        diffs[i]=100*(a[ix].mean()-b[ix].mean())
    lo,hi=np.percentile(diffs,[2.5,97.5])
    tail=np.sum(diffs<=0) if point>=0 else np.sum(diffs>=0)
    p=min(1.0,2*(tail+1)/(B+1))
    return point,lo,hi,p

metrics=[]
tests=[]

for d in ['amstertime','msls','nordland']:
    if d=='msls':
        idx=EVAL
    else:
        idx=np.arange(cos_visual[d].shape[0])

    V=cos_visual[d][idx]
    L=language[d][idx]
    P=positive[d][idx]

    visual_ret=topk_idx(V,10)
    pwf_ret,_=pwf_retrieve(V,L,rho_star,mc_star)
    const_ret=const_top10(V,L,alpha_star)
    sg_ret=sg_retrieve(V,L,tau_star,theta_star)

    methods={
        'Visual-only':visual_ret,
        'PWF':pwf_ret,
        f'TunedConstTop10_a={alpha_star:.2f}':const_ret,
        f'TunedSG_tau={tau_star:g}_theta={theta_star:.3f}':sg_ret,
    }

    for name,ret in methods.items():
        r1,r5,r10=recalls(ret,P)
        metrics.append([d,name,len(idx),r1,r5,r10])

    # Reviewer-control family: PWF vs tuned constant and tuned SG, R@1/R@5.
    for cname,cret in [('TunedConstTop10',const_ret),('TunedSG',sg_ret)]:
        for k in [1,5]:
            ca=correct_at_k(pwf_ret,P,k)
            cb=correct_at_k(cret,P,k)
            diff,lo,hi,praw=paired_boot(ca,cb)
            tests.append([d,cname,f'R@{k}',diff,lo,hi,praw,int((ca!=cb).sum()),
                          int(ca.sum()),int(cb.sum())])

metrics=pd.DataFrame(metrics,columns=['dataset','method','n','R1','R5','R10'])
tests=pd.DataFrame(tests,columns=[
    'dataset','comparison','metric','PWF_minus_control_pp','CI_low','CI_high',
    'p_raw','discordant','PWF_correct','control_correct'
])
tests['BH_FDR_q_12']=bh(tests.p_raw)

metrics.to_csv(OUT/'cosplace_second_backbone_metrics.csv',index=False)
tests.to_csv(OUT/'cosplace_second_backbone_control_BH12.csv',index=False)

selection={
    'visual_backbone':'CosPlace ResNet18 512-D',
    'image_size':'320x320',
    'MSLS_calibration_queries':len(CAL),
    'seed':int(split['seed']),
    'PWF_rho':rho_star,
    'PWF_Mc':mc_star,
    'constant_alpha':alpha_star,
    'SG_tau':tau_star,
    'SG_theta':theta_star,
}
with open(OUT/'cosplace_second_backbone_selection.json','w') as f:
    json.dump(selection,f,indent=2)

print("\nSELECTION")
print(json.dumps(selection,indent=2))
print("\nMETRICS")
display(metrics)
print("\nPAIRED CONTROL TESTS")
display(tests)

print("\nSaved all outputs to:", OUT)
